[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juliopez/Taller-Fundamentos-Data-Science-Python/blob/docencia/nb06-modelos-preentrenados-2026/Machine_Learning_2026/03_Notebooks/NB06_Transfer_Learning_Fine_Tuning_Evaluacion_CNN.ipynb)
# NB06 — Evaluación de modelos preentrenados con imágenes

## Propósito
Experimentar con **MobileNetV2** (entrenada en ImageNet) **sin entrenar ni ajustar pesos**. Primero analizaremos un conjunto común de **20 imágenes CIFAR-10 de GitHub** (dos por categoría); después **deberás** repetir el experimento con tu propio dataset del NB05 y probar una imagen individual subida a Colab.

**Objetivos:** observar predicciones y probabilidades, distinguir aciertos/errores con criterio semántico y reconocer limitaciones de resolución y de clases. **No se realiza Fine-Tuning.**

**Importante:** CIFAR-10 tiene diez categorías; ImageNet tiene mil clases que no corresponden necesariamente a las mismas etiquetas. Una coincidencia textual exacta no es una medida adecuada de acierto.

## 1. Preparar el entorno

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from pathlib import Path
from PIL import Image
from IPython.display import display
import urllib.request

print("TensorFlow:", tf.__version__)

## 2. Descargar el conjunto de ejemplo desde GitHub

El notebook descarga directamente las **20 imágenes PNG**, organizadas en diez carpetas por categoría, desde el repositorio del curso. No necesitas descargar ZIP ni conectar Google Drive para esta primera parte.

Fuente: [diccionario de datos NB06_CIFAR10](https://github.com/juliopez/Taller-Fundamentos-Data-Science-Python/blob/main/Machine_Learning_2026/02_Datasets/NB06_CIFAR10/README.md).

In [ ]:
BASE_RAW = "https://raw.githubusercontent.com/juliopez/Taller-Fundamentos-Data-Science-Python/main/Machine_Learning_2026/02_Datasets/NB06_CIFAR10"
CLASES_CIFAR10 = ["avion", "automovil", "pajaro", "gato", "ciervo", "perro", "rana", "caballo", "barco", "camion"]
DESTINO = Path("/content/NB06_CIFAR10")
registros = []
for clase in CLASES_CIFAR10:
    carpeta = DESTINO / clase
    carpeta.mkdir(parents=True, exist_ok=True)
    for numero in (1, 2):
        nombre = f"{clase}_{numero:02d}.png"
        ruta = carpeta / nombre
        if not ruta.exists():
            urllib.request.urlretrieve(f"{BASE_RAW}/{clase}/{nombre}", ruta)
        registros.append({"categoria_referencia": clase, "ruta": str(ruta), "imagen": nombre})
muestra = pd.DataFrame(registros)
assert len(muestra) == 20 and muestra["categoria_referencia"].nunique() == 10
print("Imágenes descargadas:", len(muestra))
display(muestra)

## 3. Observar el dataset antes de predecir

Revisa las imágenes y sus categorías originales. CIFAR-10 contiene imágenes de **32 × 32 píxeles**: al ampliarlas para MobileNetV2 no se recupera información visual perdida.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(4, 5, figsize=(14, 10))
for ax, fila in zip(axes.flat, muestra.itertuples(index=False)):
    with Image.open(fila.ruta) as im:
        ax.imshow(im.convert("RGB").resize((128, 128)))
    ax.set_title(fila.categoria_referencia + "\n" + fila.imagen, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Cargar MobileNetV2 preentrenada

Utilizaremos los pesos de ImageNet para **inferencia**. La primera ejecución descarga los pesos. No entrenaremos el modelo.

In [ ]:
mobilenet = keras.applications.MobileNetV2(weights="imagenet", include_top=True)
print("Parámetros del modelo:", mobilenet.count_params())

## 5. Ejecutar las predicciones

Cada imagen se redimensiona a 224 × 224 y se preprocesa con la función específica de MobileNetV2. Mostraremos las **tres etiquetas ImageNet más probables** y la probabilidad de la primera. Estas probabilidades no equivalen a una garantía de acierto.

In [ ]:
def inferir_mobilenet(ruta):
    with Image.open(ruta) as im:
        x = np.asarray(im.convert("RGB").resize((224, 224)), dtype=np.float32)[None, ...]
    x = keras.applications.mobilenet_v2.preprocess_input(x)
    probabilidades = mobilenet.predict(x, verbose=0)
    return keras.applications.mobilenet_v2.decode_predictions(probabilidades, top=3)[0]

def predecir_tabla(datos):
    filas = []
    for fila in datos.itertuples(index=False):
        preds = inferir_mobilenet(fila.ruta)
        filas.append({
            "categoria_referencia": fila.categoria_referencia,
            "imagen": fila.imagen,
            "prediccion_top1": preds[0][1],
            "probabilidad_top1": float(preds[0][2]),
            "top3": " | ".join(f"{etiqueta} ({prob:.1%})" for _, etiqueta, prob in preds),
        })
    return pd.DataFrame(filas)

resultados = predecir_tabla(muestra)
display(resultados)

## 6. Analizar aciertos y errores

Observa las imágenes y compara su categoría de referencia con la etiqueta ImageNet. **No uses una igualdad automática de nombres:** ImageNet puede emplear categorías más específicas o diferentes. Clasifica manualmente cada resultado como **coherente**, **parcialmente relacionado** o **incorrecto**, justificando tu criterio.

La siguiente tabla permite anotar tu evaluación sin modificar las predicciones.

In [ ]:
analisis = resultados[["categoria_referencia", "imagen", "prediccion_top1", "probabilidad_top1"]].copy()
analisis["evaluacion"] = ""  # coherente / parcialmente relacionado / incorrecto
analisis["justificacion"] = ""
display(analisis)
print("Completa las dos últimas columnas según tu análisis; no se calcula una exactitud automática.")

## 7. Actividad autónoma obligatoria — dataset propio del NB05

**Todos los estudiantes deben realizar esta actividad.** Monta Google Drive y configura la carpeta de imágenes que utilizaste en NB05. Debe contener subcarpetas por categoría, como en el notebook anterior.

Se seleccionarán **hasta dos imágenes por categoría**, sin exigir un mínimo fijo de categorías o imágenes. Analiza las predicciones y contrástalas con las etiquetas de tus carpetas.

**Importante:** cambia `DATASET_DIR` por la ruta real de tu dataset antes de ejecutar.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATASET_DIR = Path("/content/drive/MyDrive/MachineLearning2026/dataset")  # AJUSTAR a tu NB05
assert DATASET_DIR.is_dir(), f"No existe la carpeta: {DATASET_DIR}"
EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
propios = []
for carpeta in sorted(p for p in DATASET_DIR.iterdir() if p.is_dir()):
    imagenes = sorted(p for p in carpeta.rglob("*") if p.is_file() and p.suffix.lower() in EXT)
    for ruta in imagenes[:2]:
        propios.append({"categoria_referencia": carpeta.name, "ruta": str(ruta), "imagen": ruta.name})
assert propios, "No se encontraron imágenes compatibles en las carpetas de clases"
muestra_propia = pd.DataFrame(propios)
resultados_propios = predecir_tabla(muestra_propia)
print("Imágenes propias evaluadas:", len(muestra_propia))
display(resultados_propios)

analisis_propio = resultados_propios[["categoria_referencia", "imagen", "prediccion_top1", "probabilidad_top1"]].copy()
analisis_propio["evaluacion"] = ""  # coherente / parcialmente relacionado / incorrecto
analisis_propio["justificacion"] = ""
display(analisis_propio)

## 8. Subir una imagen individual desde tu computador

Esta actividad permite probar MobileNetV2 con **una imagen nueva**, sin agregarla al dataset ni volver a entrenar el modelo.

1. Ejecuta la siguiente celda y selecciona una imagen JPG, PNG, BMP o WEBP.
2. Observa la imagen y sus **tres predicciones ImageNet más probables**.
3. Explica en una celda Markdown si la clasificación tiene sentido y por qué una probabilidad alta no garantiza un acierto.

In [ ]:
from google.colab import files

archivos_subidos = files.upload()
assert archivos_subidos, "No se seleccionó ninguna imagen"
nombre_archivo = next(iter(archivos_subidos))
ruta_individual = Path("/content") / nombre_archivo
with Image.open(ruta_individual) as im:
    imagen_individual = im.convert("RGB")
    display(imagen_individual.copy().resize((224, 224)))

predicciones_individuales = inferir_mobilenet(ruta_individual)
tabla_individual = pd.DataFrame([
    {"posicion": i, "etiqueta_imagenet": etiqueta, "probabilidad": float(prob)}
    for i, (_, etiqueta, prob) in enumerate(predicciones_individuales, start=1)
])
print("Imagen analizada:", nombre_archivo)
display(tabla_individual)

## 9. Preguntas para responder en celdas Markdown

**A.** Identifica dos imágenes de CIFAR-10: ¿qué predijo MobileNetV2 y con qué probabilidad? ¿Son coherentes las predicciones?

**B.** Describe un caso en que la etiqueta ImageNet sea más específica o distinta de la categoría CIFAR-10.

**C.** ¿Cómo influye la resolución original de 32 × 32 píxeles? ¿Por qué redimensionar no recupera información perdida?

**D.** **Dataset propio del NB05 (obligatorio):** identifica dos predicciones, evalúa su coherencia con las categorías reales y compara con el conjunto CIFAR-10.

**E.** **Imagen individual:** indica qué imagen subiste, cuáles fueron las predicciones y si el resultado tiene sentido.

**F.** ¿Por qué una probabilidad alta no garantiza un acierto? ¿Cómo podría ayudar el *Transfer Learning* a adaptar un modelo a tus categorías, sin implementarlo?

## 10. Cierre y continuidad

En este NB06 has evaluado **MobileNetV2 preentrenada** con un conjunto común, **tu dataset del NB05** y una **imagen individual subida a Colab**, sin reentrenar pesos.

**NB07:** preparación e interoperabilidad de un modelo común de piedra, papel o tijera. **NB08:** despliegue del modelo con Streamlit.